In [ ]:
import torch
from itertools import product
from pathlib import Path
!rm -r /root/Architectural-Biases-in-Time-Series-Anomaly-Detection
!git clone -b end_to_end https://github.com/KirillVishnyakov/Architectural-Biases-in-Time-Series-Anomaly-Detection
!git -C /root/Architectural-Biases-in-Time-Series-Anomaly-Detection log --oneline -1

rm: cannot remove '/root/Architectural-Biases-in-Time-Series-Anomaly-Detection': No such file or directory
Cloning into 'Architectural-Biases-in-Time-Series-Anomaly-Detection'...
remote: Enumerating objects: 654, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 654 (delta 2), reused 21 (delta 1), pack-reused 617 (from 1)
Receiving objects: 100% (654/654), 213.12 MiB | 28.85 MiB/s, done.
Resolving deltas: 100% (330/330), done.
a4097ae (HEAD -> end_to_end, origin/end_to_end) structure


In [2]:
import sys
root_dir = "/root/Architectural-Biases-in-Time-Series-Anomaly-Detection"
sys.path.append(root_dir)
from app.utils.config import Config
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = Config(
    root_dir,
    "app",
    "data",
    "training_artifacts",
    "saved_model_weights",
    "training_results"
)
cfg.data_dir = Path("/mnt/data")
from app.training.train import fit
from app.models.transformer_encoder_forecaster import patch_transformer
from app.data.dataset import forecasting_Dataset
from app.scoring.knn_scorer import KNNResidualScorer
from app.evaluation.evaluate import evaluate_model

In [3]:
scorers = [
    KNNResidualScorer(device=device, n_components=n, k=k, chunk_size = 8000) 
    for n, k in product([0.75, 0.85, 0.95], [3, 5, 7])
]

def tune_over_grid(hyperparam_grid, stop_at_epoch = 4, scorers = scorers):
    for lr, lookback_window, d_model, num_blocks, horizon, num_heads, dropout in product(
        hyperparam_grid["lr"],
        hyperparam_grid["lookback_window"],
        hyperparam_grid["d_model"],
        hyperparam_grid["num_blocks"],
        hyperparam_grid["horizon"],
        hyperparam_grid["num_heads"],
        hyperparam_grid["dropout"]
    ):

        model = patch_transformer(
                    lookback_window, horizon, d_model,
                    num_heads, dropout, num_blocks
                ).to(device)
        model = torch.compile(model)

        name = (
            f"lr_{lr:.0e}_lw_{lookback_window}_dm_{d_model}_"
            f"nb_{num_blocks}_h_{horizon}_nh_{num_heads}_do_{dropout:.2f}"
        )

        forecast_train_dataset = forecasting_Dataset(
            device, cfg, lookback_window, horizon, start = 0, end = 800000
        )

        forecast_val_dataset = forecasting_Dataset(
            device, cfg, lookback_window, horizon, scaler = forecast_train_dataset.scaler,
            start = 800000, end = 1000000, train = False
        )

        fit(
            device=device,
            model=model,
            name=name,
            train_dataset=forecast_train_dataset,
            test_dataset=forecast_val_dataset,
            lr=lr,
            batch_size=512,
            num_epochs=10,
            stop_epoch_ratio=stop_at_epoch / 10.0,
            shuffle = True,
            patience = 5,
            min_delta = 0.0001
        )

        scoring_val_dataset = forecasting_Dataset(
            device, cfg, lookback_window, horizon, scaler = forecast_train_dataset.scaler,
            start = 1000000, end = 3500000, train = False
        )

        evaluate_model(device, model, forecast_val_dataset, scoring_val_dataset, name, cfg, scorers)


### horizons

In [4]:
hyperparam_grid = {
    "lr": [0.001],
    "lookback_window": [256],
    "d_model": [256],
    "num_blocks": [1],
    "horizon": [4, 8, 16],
    "num_heads": [8],
    "dropout": [0.0]
}
tune_over_grid(hyperparam_grid)

32
LR Scheduler: 1093 warmup steps, 15620 total steps
|lr_1e-03_lw_256_dm_256_nb_1_h_4_nh_8_do_0.00| train = 0.1667 | test= 0.0214 | LR: 9.97e-04
|lr_1e-03_lw_256_dm_256_nb_1_h_4_nh_8_do_0.00| train = 0.0215 | test= 0.0206 | LR: 9.53e-04
|lr_1e-03_lw_256_dm_256_nb_1_h_4_nh_8_do_0.00| train = 0.0211 | test= 0.0206 | LR: 8.58e-04
|lr_1e-03_lw_256_dm_256_nb_1_h_4_nh_8_do_0.00| train = 0.0209 | test= 0.0204 | LR: 7.23e-04
--- evaluating model lr_1e-03_lw_256_dm_256_nb_1_h_4_nh_8_do_0.00 --- 
32
LR Scheduler: 1093 warmup steps, 15620 total steps
|lr_1e-03_lw_256_dm_256_nb_1_h_8_nh_8_do_0.00| train = 0.1615 | test= 0.0339 | LR: 9.97e-04
|lr_1e-03_lw_256_dm_256_nb_1_h_8_nh_8_do_0.00| train = 0.0344 | test= 0.0333 | LR: 9.53e-04
|lr_1e-03_lw_256_dm_256_nb_1_h_8_nh_8_do_0.00| train = 0.0339 | test= 0.0335 | LR: 8.58e-04
|lr_1e-03_lw_256_dm_256_nb_1_h_8_nh_8_do_0.00| train = 0.0336 | test= 0.0331 | LR: 7.23e-04
--- evaluating model lr_1e-03_lw_256_dm_256_nb_1_h_8_nh_8_do_0.00 --- 
32
LR Schedule

### Lookbacks

In [5]:
hyperparam_grid = {
    "lr": [0.001],
    "lookback_window": [128, 256, 384, 512],
    "d_model": [256],
    "num_blocks": [1],
    "horizon": [8],
    "num_heads": [8],
    "dropout": [0.0]
}
tune_over_grid(hyperparam_grid)

16
LR Scheduler: 1094 warmup steps, 15630 total steps
|lr_1e-03_lw_128_dm_256_nb_1_h_8_nh_8_do_0.00| train = 0.0764 | test= 0.0341 | LR: 9.97e-04
|lr_1e-03_lw_128_dm_256_nb_1_h_8_nh_8_do_0.00| train = 0.0344 | test= 0.0336 | LR: 9.53e-04
|lr_1e-03_lw_128_dm_256_nb_1_h_8_nh_8_do_0.00| train = 0.0338 | test= 0.0337 | LR: 8.58e-04
|lr_1e-03_lw_128_dm_256_nb_1_h_8_nh_8_do_0.00| train = 0.0335 | test= 0.0334 | LR: 7.23e-04
--- evaluating model lr_1e-03_lw_128_dm_256_nb_1_h_8_nh_8_do_0.00 --- 
32
LR Scheduler: 1093 warmup steps, 15620 total steps
|lr_1e-03_lw_256_dm_256_nb_1_h_8_nh_8_do_0.00| train = 0.1560 | test= 0.0343 | LR: 9.97e-04
|lr_1e-03_lw_256_dm_256_nb_1_h_8_nh_8_do_0.00| train = 0.0345 | test= 0.0334 | LR: 9.53e-04
|lr_1e-03_lw_256_dm_256_nb_1_h_8_nh_8_do_0.00| train = 0.0340 | test= 0.0331 | LR: 8.58e-04
|lr_1e-03_lw_256_dm_256_nb_1_h_8_nh_8_do_0.00| train = 0.0337 | test= 0.0330 | LR: 7.23e-04
--- evaluating model lr_1e-03_lw_256_dm_256_nb_1_h_8_nh_8_do_0.00 --- 
48
LR Schedule

### d_model

In [4]:
hyperparam_grid = {
    "lr": [0.001],
    "lookback_window": [128],
    "d_model": [128, 256, 384],
    "num_blocks": [1],
    "horizon": [8],
    "num_heads": [8],
    "dropout": [0.0]
}
tune_over_grid(hyperparam_grid)

16
LR Scheduler: 1094 warmup steps, 15630 total steps
|lr_1e-03_lw_128_dm_128_nb_1_h_8_nh_8_do_0.00| train = 0.0506 | test= 0.0347 | LR: 9.97e-04
|lr_1e-03_lw_128_dm_128_nb_1_h_8_nh_8_do_0.00| train = 0.0346 | test= 0.0343 | LR: 9.53e-04
|lr_1e-03_lw_128_dm_128_nb_1_h_8_nh_8_do_0.00| train = 0.0339 | test= 0.0336 | LR: 8.58e-04
|lr_1e-03_lw_128_dm_128_nb_1_h_8_nh_8_do_0.00| train = 0.0336 | test= 0.0333 | LR: 7.23e-04
--- evaluating model lr_1e-03_lw_128_dm_128_nb_1_h_8_nh_8_do_0.00 --- 
16
LR Scheduler: 1094 warmup steps, 15630 total steps
|lr_1e-03_lw_128_dm_256_nb_1_h_8_nh_8_do_0.00| train = 0.0734 | test= 0.0343 | LR: 9.97e-04
|lr_1e-03_lw_128_dm_256_nb_1_h_8_nh_8_do_0.00| train = 0.0344 | test= 0.0344 | LR: 9.53e-04
|lr_1e-03_lw_128_dm_256_nb_1_h_8_nh_8_do_0.00| train = 0.0338 | test= 0.0336 | LR: 8.58e-04
|lr_1e-03_lw_128_dm_256_nb_1_h_8_nh_8_do_0.00| train = 0.0336 | test= 0.0334 | LR: 7.23e-04
--- evaluating model lr_1e-03_lw_128_dm_256_nb_1_h_8_nh_8_do_0.00 --- 
16
LR Schedule

### blocks

In [5]:
hyperparam_grid = {
    "lr": [0.001],
    "lookback_window": [128],
    "d_model": [128],
    "num_blocks": [1, 2, 3],
    "horizon": [8],
    "num_heads": [8],
    "dropout": [0.0]
}
tune_over_grid(hyperparam_grid)

16
LR Scheduler: 1094 warmup steps, 15630 total steps


W0615 20:37:03.364000 296 site-packages/torch/_dynamo/convert_frame.py:1016] [3/8] torch._dynamo hit config.recompile_limit (8)
W0615 20:37:03.364000 296 site-packages/torch/_dynamo/convert_frame.py:1016] [3/8]    function: 'forward' (/root/Architectural-Biases-in-Time-Series-Anomaly-Detection/app/models/transformer_encoder_forecaster.py:50)
W0615 20:37:03.364000 296 site-packages/torch/_dynamo/convert_frame.py:1016] [3/8]    last reason: 3/7: GLOBAL_STATE changed: grad_mode autocast 
W0615 20:37:03.364000 296 site-packages/torch/_dynamo/convert_frame.py:1016] [3/8] To log all recompilation reasons, use TORCH_LOGS="recompiles".
W0615 20:37:03.364000 296 site-packages/torch/_dynamo/convert_frame.py:1016] [3/8] To diagnose recompilation issues, see https://pytorch.org/docs/main/torch.compiler_troubleshooting.html.
W0615 20:39:04.514000 296 site-packages/torch/_dynamo/convert_frame.py:1016] [4/8] torch._dynamo hit config.recompile_limit (8)
W0615 20:39:04.514000 296 site-packages/torch/_d

|lr_1e-03_lw_128_dm_128_nb_1_h_8_nh_8_do_0.00| train = 0.0478 | test= 0.0342 | LR: 9.97e-04
|lr_1e-03_lw_128_dm_128_nb_1_h_8_nh_8_do_0.00| train = 0.0346 | test= 0.0339 | LR: 9.53e-04
|lr_1e-03_lw_128_dm_128_nb_1_h_8_nh_8_do_0.00| train = 0.0340 | test= 0.0334 | LR: 8.58e-04
|lr_1e-03_lw_128_dm_128_nb_1_h_8_nh_8_do_0.00| train = 0.0337 | test= 0.0332 | LR: 7.23e-04
--- evaluating model lr_1e-03_lw_128_dm_128_nb_1_h_8_nh_8_do_0.00 --- 
16
LR Scheduler: 1094 warmup steps, 15630 total steps
|lr_1e-03_lw_128_dm_128_nb_2_h_8_nh_8_do_0.00| train = 0.0529 | test= 0.0341 | LR: 9.97e-04
|lr_1e-03_lw_128_dm_128_nb_2_h_8_nh_8_do_0.00| train = 0.0343 | test= 0.0338 | LR: 9.53e-04
|lr_1e-03_lw_128_dm_128_nb_2_h_8_nh_8_do_0.00| train = 0.0337 | test= 0.0336 | LR: 8.58e-04
|lr_1e-03_lw_128_dm_128_nb_2_h_8_nh_8_do_0.00| train = 0.0335 | test= 0.0334 | LR: 7.23e-04
--- evaluating model lr_1e-03_lw_128_dm_128_nb_2_h_8_nh_8_do_0.00 --- 
16
LR Scheduler: 1094 warmup steps, 15630 total steps
|lr_1e-03_lw_1

### heads

In [6]:
hyperparam_grid = {
    "lr": [0.001],
    "lookback_window": [128],
    "d_model": [128],
    "num_blocks": [2],
    "horizon": [8],
    "num_heads": [4, 8, 16],
    "dropout": [0.0]
}
tune_over_grid(hyperparam_grid)

16
LR Scheduler: 1094 warmup steps, 15630 total steps
|lr_1e-03_lw_128_dm_128_nb_2_h_8_nh_4_do_0.00| train = 0.0534 | test= 0.0344 | LR: 9.97e-04
|lr_1e-03_lw_128_dm_128_nb_2_h_8_nh_4_do_0.00| train = 0.0342 | test= 0.0336 | LR: 9.53e-04
|lr_1e-03_lw_128_dm_128_nb_2_h_8_nh_4_do_0.00| train = 0.0336 | test= 0.0336 | LR: 8.58e-04
|lr_1e-03_lw_128_dm_128_nb_2_h_8_nh_4_do_0.00| train = 0.0335 | test= 0.0333 | LR: 7.23e-04
--- evaluating model lr_1e-03_lw_128_dm_128_nb_2_h_8_nh_4_do_0.00 --- 
16
LR Scheduler: 1094 warmup steps, 15630 total steps
|lr_1e-03_lw_128_dm_128_nb_2_h_8_nh_8_do_0.00| train = 0.0520 | test= 0.0342 | LR: 9.97e-04
|lr_1e-03_lw_128_dm_128_nb_2_h_8_nh_8_do_0.00| train = 0.0342 | test= 0.0336 | LR: 9.53e-04
|lr_1e-03_lw_128_dm_128_nb_2_h_8_nh_8_do_0.00| train = 0.0336 | test= 0.0336 | LR: 8.58e-04
|lr_1e-03_lw_128_dm_128_nb_2_h_8_nh_8_do_0.00| train = 0.0334 | test= 0.0333 | LR: 7.23e-04
--- evaluating model lr_1e-03_lw_128_dm_128_nb_2_h_8_nh_8_do_0.00 --- 
16
LR Schedule

In [9]:
lr = 0.001
lookback_window = 128
d_model = 128
num_blocks = 2
horizon = 8
num_heads = 4
dropout = 0.15

forecast_train_dataset = forecasting_Dataset(
    device, cfg, lookback_window, horizon, start = 0, end = 800000
)

forecast_val_dataset = forecasting_Dataset(
    device, cfg, lookback_window, horizon, scaler = forecast_train_dataset.scaler,
    start = 800000, end = 1000000, train = False
)

scoring_val_dataset = forecasting_Dataset(
    device, cfg, lookback_window, horizon, scaler = forecast_train_dataset.scaler,
    start = 1000000, end = 3500000, train = False
)

final_model = patch_transformer(
            lookback_window, horizon, d_model,
            num_heads, dropout, num_blocks
        ).to(device)
final_model = torch.compile(final_model)

fit(
    device=device,
    model=final_model,
    name="final_model",
    train_dataset=forecast_train_dataset,
    test_dataset=forecast_val_dataset,
    lr=lr,
    batch_size = 512,
    num_epochs = 20,
    shuffle = True,
    patience = 7,
    min_delta = 0.0001
)

scorers = [
    KNNResidualScorer(device=device, n_components=n, k=k, chunk_size = 8000) 
    for n, k in product([0.75, 0.85, 0.95], [3, 5, 7])
]
evaluate_model(device, final_model, forecast_val_dataset, scoring_val_dataset, "final_model", cfg, scorers)

16
LR Scheduler: 2188 warmup steps, 31260 total steps


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x2b364ee7dc60>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1647, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x2b364ee7dc60>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", l

|final_model| train = 0.0632 | test= 0.0367 | LR: 7.14e-04
|final_model| train = 0.0363 | test= 0.0346 | LR: 9.97e-04
|final_model| train = 0.0350 | test= 0.0339 | LR: 9.82e-04
|final_model| train = 0.0344 | test= 0.0339 | LR: 9.53e-04
|final_model| train = 0.0341 | test= 0.0334 | LR: 9.11e-04
|final_model| train = 0.0339 | test= 0.0335 | LR: 8.58e-04
|final_model| train = 0.0338 | test= 0.0338 | LR: 7.95e-04
|final_model| train = 0.0337 | test= 0.0333 | LR: 7.23e-04
|final_model| train = 0.0336 | test= 0.0333 | LR: 6.45e-04
|final_model| train = 0.0335 | test= 0.0333 | LR: 5.63e-04
|final_model| train = 0.0334 | test= 0.0335 | LR: 4.80e-04
|final_model| train = 0.0334 | test= 0.0335 | LR: 3.97e-04
|final_model| train = 0.0333 | test= 0.0334 | LR: 3.18e-04
|final_model| train = 0.0332 | test= 0.0334 | LR: 2.43e-04
|final_model| train = 0.0332 | test= 0.0334 | LR: 1.76e-04
| experiment: final_model | epoch 15, train: MSE 0.0332, test MSE: 0.0334
Stopping early
--- evaluating model final

In [10]:
clean_state_dict = final_model._orig_mod.state_dict()
torch.save(clean_state_dict, "/root/Architectural-Biases-in-Time-Series-Anomaly-Detection/training_artifacts/saved_model_weights/final_model_weights.pth")